# Visualization

The earlier notebooks drew signals with `pypft.plot_signal` and `pypft.render_cartesian`. This notebook covers what they did not: how `plot_signal` chooses its color scales, and the tracing functions `pypft.forward_pft_traced` and `pypft.inverse_pft_traced`, which run the same pipeline as `pypft.forward_pft` and `pypft.inverse_pft` while keeping every intermediate signal and, on request, a figure of each one.

In [ ]:
# Draw every figure below the cell that makes it
%matplotlib inline

# NumPy for arrays, matplotlib for figures, and PyPFT itself
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import pypft

## A test image sampled on the transform's own grid

`tests/samples/maze_polar.tif` is a circular maze that `scripts/make_maze.py` builds directly in polar coordinates: a central courtyard inside eight rings of cells, with round walls between the rings and radial walls between the cells of a ring. The walls are evaluated at the sample points of a `pypft.PolarGrid` with `n_radial=576`, `n_angular=48`, and space limit $R = 1$, so the file already holds a `(radial, angular)` array on that grid and needs no resampling.

In [ ]:
# Load the maze: an 8-bit array of shape (n_radial, n_angular)
values = plt.imread(fname="../tests/samples/maze_polar.tif").astype(np.float64)

# Rebuild the grid the file was sampled on
grid = pypft.PolarGrid(n_radial=576, n_angular=48, R=1.0)

# Wrap the samples as a polar spatial signal and draw it as a disk
signal = pypft.PolarSpatialSignal(values=values, grid=grid)
maze_ax = pypft.render_cartesian(signal=signal, height=256, width=256)
maze_ax.set_title(label="maze_polar.tif")
plt.show()

## Color scales of `plot_signal`

`plot_signal` draws every domain the same way, as a magnitude panel and a phase panel:

- The magnitude uses a power-law color scale with exponent $\gamma = 0.3$, so a pixel of magnitude $m$ is colored by $(m / m_{\max})^{\gamma}$. A transform's magnitude is usually dominated by a few large values near the center; the exponent lifts the small values so the rest of the plane stays visible, and the colorbar still shows the true magnitudes.
- The magnitude of a `POLAR_SPATIAL` signal is drawn in grayscale, since it is an image; every other domain uses matplotlib's default colormap.
- The phase uses the cyclic `twilight` colormap with its range fixed to $[-\pi, \pi]$, so $-\pi$ and $\pi$ get the same color and a constant phase (such as the zero phase of a real image) is drawn as one flat color.

The cell below plots the maze's magnitude with a plain linear scale next to `plot_signal`'s own panels, in the frequency domain where the difference is largest.

In [ ]:
# Transform the maze to the polar frequency domain
frequency_signal = signal.to(domain=pypft.Domain.POLAR_FREQUENCY)

# Left: a plain linear color scale of the magnitude
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4))
image = axes[0].imshow(X=np.abs(frequency_signal.values), aspect="auto")
fig.colorbar(mappable=image, ax=axes[0])
axes[0].set_title(label="linear magnitude")
axes[0].set_xlabel(xlabel="angular index")
axes[0].set_ylabel(ylabel="radial index")

# Middle and right: plot_signal's gamma-enhanced magnitude and its phase
frequency_signal.plot(ax=(axes[1], axes[2]))
fig.tight_layout()
plt.show()

## Tracing a transform

`forward_pft_traced` returns a `pypft.PFTTrace`. Its `values` equal what `forward_pft` returns, and its `signals` hold the signal in each of the four domains, in the order visited. With `visualize_steps=True` the trace also builds one figure per domain, each drawn by `plot_signal`; `figures` holds them and `figure_labels` names them.

In [ ]:
# Trace the forward transform, with one figure per domain
trace = pypft.forward_pft_traced(f=values, grid=grid, visualize_steps=True)

# The traced result is the plain forward transform
expected = pypft.forward_pft(f=values, grid=grid)
print(f"largest difference from forward_pft: {np.abs(trace.values - expected).max():.1e}")

# The domains visited, and the label of each figure
for visited, label in zip(trace.signals, trace.figure_labels):
    print(f"{visited.domain.name:<26} {label}")
plt.show()

`inverse_pft_traced` walks the same chain backward. Its first signal is the frequency-domain input and its last is the reconstructed image, which `render_cartesian` can draw next to the original.

In [ ]:
# Trace the inverse transform without building figures
inverse_trace = pypft.inverse_pft_traced(F=trace.values, grid=grid)
print(f"round-trip error: {np.abs(inverse_trace.values - values).max():.1e}")

# Draw the two endpoints of the inverse trace as disks
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
pypft.render_cartesian(signal=inverse_trace.signals[0], height=256, width=256, ax=axes[0])
axes[0].set_title(label="first signal: polar frequency")
pypft.render_cartesian(signal=inverse_trace.signals[-1], height=256, width=256, ax=axes[1])
axes[1].set_title(label="last signal: reconstructed image")
fig.tight_layout()
plt.show()

## The pipeline figure

`visualize_pipeline=True` builds a single figure holding the magnitude and phase panels of every domain. It can be combined with `visualize_steps=True`; each keyword adds its own figures to the same `PFTTrace`.

In [ ]:
# Trace the forward transform again, as one pipeline figure
pipeline_trace = pypft.forward_pft_traced(f=values, grid=grid, visualize_pipeline=True)
print(pipeline_trace.figure_labels)
plt.show()

## Saving and closing figures

`PFTTrace.save` writes every figure of a trace as a PNG file into a directory, creating it if needed, and names each file by its position and label. Tracing never writes to disk on its own. `PFTTrace.close` closes every figure of a trace; matplotlib warns once more than 20 figures are open, so close traces you are done with. In a notebook, `plt.show()` already closes the figures it displays; a trace whose figures are never shown keeps them open until `close` is called. The cell below saves the step figures into a new temporary directory; change `output_directory` to the path you want them saved in.

In [ ]:
# The directory the figures are saved in: change this to your own path
output_directory = Path(tempfile.mkdtemp(prefix="pypft_figures_"))

# Save the step figures and list the directory and the saved files
paths = trace.save(directory=output_directory)
print(f"saved into: {output_directory}")
print([path.name for path in paths])

# Trace once more without showing the figures: they stay open until closed
quiet_trace = pypft.forward_pft_traced(f=values, grid=grid, visualize_steps=True)
print(f"open figures after tracing: {len(plt.get_fignums())}")

# Close every figure of the trace
quiet_trace.close()
print(f"open figures after closing: {len(plt.get_fignums())}")